# One-time dedupe of silver lab_results
Removes the duplicate copies of each `lab_result_id` created by the old APPEND loads, keeping the latest version: latest `reported_at`, then latest `insert_timestamp` (silver load time). Run once before switching to `bronze_to_silver_upsert`.

The `silver_schema` widget defaults to `silver_test` so it can't hit real silver by accident. Set it to `silver` for the real run. The previous version can be restored with Delta time travel (version printed below).

In [0]:
dbutils.widgets.text("silver_schema", "silver_test", "Silver Schema")
silver_schema = dbutils.widgets.get("silver_schema").strip()
target_table = f"adb_healthcare.{silver_schema}.lab_results"

print("=" * 70)
print(f"[START] one-time dedupe on {target_table}")
version_before = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").collect()[0]["version"]
print(f"[BEFORE] Delta version (restore point): {version_before}")

stats_query = f"SELECT count(*) AS n_rows, count(DISTINCT lab_result_id) AS n_keys FROM {target_table}"
before = spark.sql(stats_query).collect()[0]
print(f"[BEFORE] rows = {before['n_rows']}, distinct lab_result_id = {before['n_keys']}")

# Safety check: the delete matches rows on (lab_result_id, landing_timestamp, insert_timestamp),
# so that combination must identify exactly one row
non_unique = spark.sql(f"""
SELECT count(*) AS n FROM (
  SELECT lab_result_id, landing_timestamp, insert_timestamp
  FROM {target_table}
  GROUP BY ALL HAVING count(*) > 1)
""").collect()[0]["n"]
if non_unique > 0:
    raise ValueError(f"{non_unique} rows are not unique on (lab_result_id, landing_timestamp, insert_timestamp) - stopping")
print("[CHECK] Rows are unique on (lab_result_id, landing_timestamp, insert_timestamp)")

In [0]:
# Delete only the rows that are NOT the latest version of their lab_result_id
delete_query = f"""
MERGE INTO {target_table} AS t
USING (
  SELECT lab_result_id, landing_timestamp, insert_timestamp
  FROM {target_table}
  QUALIFY row_number() OVER (
    PARTITION BY lab_result_id
    ORDER BY reported_at DESC NULLS LAST, insert_timestamp DESC) > 1
) AS d
  ON  t.lab_result_id = d.lab_result_id
  AND t.landing_timestamp <=> d.landing_timestamp
  AND t.insert_timestamp <=> d.insert_timestamp
WHEN MATCHED THEN DELETE
"""
print("[DELETE] Query prepared:")
print("-" * 70)
print(delete_query.strip())
print("-" * 70)
result = spark.sql(delete_query).collect()[0].asDict()
print(f"[DELETE] Rows deleted: {result.get('num_deleted_rows')}")

In [0]:
after = spark.sql(stats_query).collect()[0]
print(f"[AFTER] rows = {after['n_rows']}, distinct lab_result_id = {after['n_keys']}")
if after["n_rows"] != after["n_keys"] or after["n_keys"] != before["n_keys"]:
    raise ValueError(f"Unexpected result - restore with: RESTORE TABLE {target_table} TO VERSION AS OF {version_before}")
print(f"[OK] One row per lab_result_id, no test lost. Restore point if needed: version {version_before}")
print("=" * 70)